In [1]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# Use the kagglehub client library to attach Kaggle resources like competitions, datasets, and models to your session
# Learn more about kagglehub: https://github.com/Kaggle/kagglehub/blob/main/README.md

import kagglehub
# kagglehub.dataset_download('<owner>/<dataset-slug>')

In [12]:
import os
import re
import json
import csv
import time
import random
from pathlib import Path
from collections import defaultdict, Counter

import numpy as np
import rasterio

import torch
import torch.nn as nn
import torch.optim as optim

from torch.utils.data import Dataset, DataLoader

print("PyTorch:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))

PyTorch: 2.11.0+cu128
CUDA available: True
GPU: Tesla T4


In [13]:
TS_ROOT = Path(
    "/kaggle/input/datasets/z789456sx/ts-satfire/ts-satfire"
)

print("TS-SatFire root:")
print(TS_ROOT)

print("\nExists:", TS_ROOT.exists())

TS-SatFire root:
/kaggle/input/datasets/z789456sx/ts-satfire/ts-satfire

Exists: True


In [14]:
day_files = list(
    TS_ROOT.rglob("*_VIIRS_Day.tif")
)

night_files = list(
    TS_ROOT.rglob("*_VIIRS_Night.tif")
)

print("Day files   :", len(day_files))
print("Night files :", len(night_files))


date_pattern = re.compile(
    r"(\d{4}-\d{2}-\d{2})"
)


day_index = defaultdict(dict)
night_index = defaultdict(dict)


for path in day_files:

    event_id = path.parent.parent.name

    match = date_pattern.search(path.name)

    if match:
        date = match.group(1)
        day_index[event_id][date] = path


for path in night_files:

    event_id = path.parent.parent.name

    match = date_pattern.search(path.name)

    if match:
        date = match.group(1)
        night_index[event_id][date] = path


events = {}

for event_id in sorted(day_index.keys()):

    common_dates = sorted(
        set(day_index[event_id]) &
        set(night_index[event_id])
    )

    if common_dates:
        events[event_id] = common_dates


SEQUENCE_LENGTH = 10


eligible_events = {
    event_id: dates
    for event_id, dates in events.items()
    if len(dates) >= SEQUENCE_LENGTH
}


print("\n========== DATASET INDEX ==========")

print("Total events found      :", len(events))

print(
    "Eligible events (>=10) :",
    len(eligible_events)
)

total_observations = sum(
    len(dates)
    for dates in eligible_events.values()
)

total_windows = sum(
    len(dates) - SEQUENCE_LENGTH + 1
    for dates in eligible_events.values()
)

print(
    "Usable observations     :",
    total_observations
)

print(
    "10-frame windows        :",
    total_windows
)

Day files   : 3556
Night files : 3508

========== DATASET INDEX ==========
Total events found      : 178
Eligible events (>=10) : 146
Usable observations     : 3275
10-frame windows        : 1961


In [16]:
# =========================================================
# Recreate the locked event split
# =========================================================

SEED = 42

random.seed(SEED)

events_by_year = defaultdict(list)

for event_id, dates in eligible_events.items():

    year = dates[0][:4]

    events_by_year[year].append(
        event_id
    )


# Shuffle within each year
for year in events_by_year:

    random.shuffle(
        events_by_year[year]
    )


# ---------------------------------------------------------
# Create proportional 70/15/15 split
# ---------------------------------------------------------

train_events = []
val_events = []
test_events = []


for year in sorted(events_by_year):

    ids = events_by_year[year]

    n = len(ids)


    n_train = round(
        n * 0.70
    )

    n_val = round(
        n * 0.15
    )


    # Keep at least one event for test
    n_train = min(
        n_train,
        n - 2
    )

    n_val = min(
        n_val,
        n - n_train - 1
    )


    train_events.extend(
        ids[:n_train]
    )

    val_events.extend(
        ids[
            n_train:
            n_train + n_val
        ]
    )

    test_events.extend(
        ids[
            n_train + n_val:
        ]
    )


# Sort for reproducibility
train_events = sorted(train_events)
val_events = sorted(val_events)
test_events = sorted(test_events)


# ---------------------------------------------------------
# Check split
# ---------------------------------------------------------

print("Locked split recreated:")
print()

print(
    "Train events:",
    len(train_events)
)

print(
    "Val events  :",
    len(val_events)
)

print(
    "Test events :",
    len(test_events)
)

print(
    "Total events:",
    len(train_events)
    + len(val_events)
    + len(test_events)
)


# ---------------------------------------------------------
# Calculate temporal windows
# ---------------------------------------------------------

def count_windows(
    event_ids,
    events_dict,
    sequence_length=10
):

    return sum(
        max(
            0,
            len(events_dict[event_id])
            - sequence_length
            + 1
        )
        for event_id in event_ids
    )


train_windows = count_windows(
    train_events,
    eligible_events,
    SEQUENCE_LENGTH
)

val_windows = count_windows(
    val_events,
    eligible_events,
    SEQUENCE_LENGTH
)

test_windows = count_windows(
    test_events,
    eligible_events,
    SEQUENCE_LENGTH
)


print()
print("Temporal windows:")
print(
    "Train:",
    train_windows
)

print(
    "Val  :",
    val_windows
)

print(
    "Test :",
    test_windows
)

print(
    "Total:",
    train_windows
    + val_windows
    + test_windows
)


# ---------------------------------------------------------
# Year distribution
# ---------------------------------------------------------

def year_distribution(
    event_ids,
    events_dict
):

    counts = Counter(
        events_dict[event_id][0][:4]
        for event_id in event_ids
    )

    return dict(
        sorted(counts.items())
    )


print()
print("Year distribution:")

print(
    "Train:",
    year_distribution(
        train_events,
        eligible_events
    )
)

print(
    "Val  :",
    year_distribution(
        val_events,
        eligible_events
    )
)

print(
    "Test :",
    year_distribution(
        test_events,
        eligible_events
    )
)

Locked split recreated:

Train events: 101
Val events  : 21
Test events : 24
Total events: 146

Temporal windows:
Train: 1355
Val  : 236
Test : 370
Total: 1961

Year distribution:
Train: {'2017': 20, '2018': 23, '2019': 6, '2020': 32, '2021': 19, '2022': 1}
Val  : {'2017': 4, '2018': 5, '2019': 1, '2020': 7, '2021': 4}
Test : {'2017': 4, '2018': 5, '2019': 2, '2020': 7, '2021': 4, '2022': 2}


In [17]:
SPLIT_FILE = Path(
    "/kaggle/working/ts_satfire_split.json"
)

split_data = {
    "seed": SEED,
    "sequence_length": SEQUENCE_LENGTH,
    "train_events": train_events,
    "val_events": val_events,
    "test_events": test_events
}

with open(
    SPLIT_FILE,
    "w"
) as f:

    json.dump(
        split_data,
        f,
        indent=2
    )


print(
    "Split saved to:"
)

print(SPLIT_FILE)

print(
    "\nExists:",
    SPLIT_FILE.exists()
)

Split saved to:
/kaggle/working/ts_satfire_split.json

Exists: True


In [18]:
SPLIT_FILE = Path(
    "/kaggle/working/ts_satfire_split.json"
)

print("Split file exists:", SPLIT_FILE.exists())

with open(SPLIT_FILE, "r") as f:
    split_data = json.load(f)


train_events = split_data["train_events"]
val_events = split_data["val_events"]
test_events = split_data["test_events"]

SEQUENCE_LENGTH = split_data["sequence_length"]


print("\nLocked split:")
print("Train events:", len(train_events))
print("Val events  :", len(val_events))
print("Test events :", len(test_events))
print("Sequence length:", SEQUENCE_LENGTH)

Split file exists: True

Locked split:
Train events: 101
Val events  : 21
Test events : 24
Sequence length: 10


In [19]:
MEAN = np.array([
    18.76488,
    27.441864,
    20.584806,
    305.99478,
    294.31738,
    14.625097,
    276.4207,
    275.16766
], dtype=np.float32)


STD = np.array([
    15.911591,
    14.879259,
    10.832616,
    21.761852,
    24.703484,
    9.878246,
    40.64329,
    40.7657
], dtype=np.float32)


print("Normalization vectors:")
print("Mean:", MEAN)
print("Std :", STD)

Normalization vectors:
Mean: [ 18.76488   27.441864  20.584806 305.99478  294.31738   14.625097
 276.4207   275.16766 ]
Std : [15.911591 14.879259 10.832616 21.761852 24.703484  9.878246 40.64329
 40.7657  ]


In [20]:
def build_event_files(event_ids):

    samples = []

    for event_id in event_ids:

        event_root = TS_ROOT / event_id

        day_dir = event_root / "VIIRS_Day"
        night_dir = event_root / "VIIRS_Night"

        if not day_dir.exists():
            continue

        if not night_dir.exists():
            continue


        day_files = {
            p.stem.replace(
                "_VIIRS_Day",
                ""
            ): p

            for p in day_dir.glob(
                "*_VIIRS_Day.tif"
            )
        }


        night_files = {
            p.stem.replace(
                "_VIIRS_Night",
                ""
            ): p

            for p in night_dir.glob(
                "*_VIIRS_Night.tif"
            )
        }


        common_dates = sorted(
            set(day_files.keys()) &
            set(night_files.keys())
        )


        for start in range(
            len(common_dates) - SEQUENCE_LENGTH + 1
        ):

            dates = common_dates[
                start:
                start + SEQUENCE_LENGTH
            ]


            samples.append({

                "event_id": event_id,

                "dates": dates,

                "day_files": [
                    day_files[d]
                    for d in dates
                ],

                "night_files": [
                    night_files[d]
                    for d in dates
                ]

            })


    return samples

In [21]:
class TSSatFireDataset(Dataset):

    def __init__(
        self,
        event_ids,
        crop_size=256,
        crop_offset=128
    ):

        self.crop_size = crop_size
        self.crop_offset = crop_offset

        self.samples = build_event_files(
            event_ids
        )

        print(
            f"Created dataset: "
            f"{len(self.samples)} temporal windows "
            f"from {len(event_ids)} events"
        )


    def __len__(self):

        return len(self.samples)


    def __getitem__(self, idx):

        sample = self.samples[idx]

        X_sequence = []
        Y_sequence = []


        for day_path, night_path in zip(
            sample["day_files"],
            sample["night_files"]
        ):


            # =============================================
            # DAY
            # =============================================

            with rasterio.open(day_path) as src:

                day = src.read().astype(
                    np.float32
                )


            # First six Day bands = input
            day_input = day[:6]

            # Day band 7 = active-fire target
            active_fire = day[6]


            # =============================================
            # NIGHT
            # =============================================

            with rasterio.open(night_path) as src:

                night = src.read().astype(
                    np.float32
                )


            # IMPORTANT:
            # Always use exactly two Night bands.
            night_input = night[:2]


            # =============================================
            # Check channels
            # =============================================

            if day_input.shape[0] != 6:

                raise ValueError(
                    f"Unexpected Day channel count: "
                    f"{day_input.shape[0]}\n"
                    f"File: {day_path}"
                )


            if night_input.shape[0] != 2:

                raise ValueError(
                    f"Unexpected Night channel count: "
                    f"{night_input.shape[0]}\n"
                    f"File: {night_path}"
                )


            # =============================================
            # Crop
            # =============================================

            y0 = self.crop_offset
            x0 = self.crop_offset

            y1 = y0 + self.crop_size
            x1 = x0 + self.crop_size


            day_input = day_input[
                :,
                y0:y1,
                x0:x1
            ]


            night_input = night_input[
                :,
                y0:y1,
                x0:x1
            ]


            active_fire = active_fire[
                y0:y1,
                x0:x1
            ]


            # =============================================
            # Combine 6 Day + 2 Night
            # =============================================

            X = np.concatenate(
                [
                    day_input,
                    night_input
                ],
                axis=0
            )


            if X.shape[0] != 8:

                raise ValueError(
                    f"Expected 8 channels, "
                    f"got {X.shape[0]}\n"
                    f"Event: {sample['event_id']}\n"
                    f"Date: {sample['dates']}"
                )


            # =============================================
            # Clean input
            # =============================================

            X = np.nan_to_num(
                X,
                nan=0.0,
                posinf=0.0,
                neginf=0.0
            )


            # =============================================
            # Active-fire target
            # =============================================

            Y = (
                np.isfinite(active_fire)
                &
                (active_fire > 0)
            ).astype(np.float32)


            # =============================================
            # Normalize
            # =============================================

            X = (
                X -
                MEAN[:, None, None]
            ) / (
                STD[:, None, None] +
                1e-8
            )


            X_sequence.append(X)
            Y_sequence.append(Y)


        # =============================================
        # Stack temporal dimension
        # =============================================

        X_sequence = np.stack(
            X_sequence,
            axis=0
        ).astype(np.float32)


        Y_sequence = np.stack(
            Y_sequence,
            axis=0
        ).astype(np.float32)


        return (
            torch.from_numpy(X_sequence),
            torch.from_numpy(Y_sequence)
        )

In [22]:
BATCH_SIZE = 2


train_dataset = TSSatFireDataset(
    train_events
)

val_dataset = TSSatFireDataset(
    val_events
)

test_dataset = TSSatFireDataset(
    test_events
)


train_loader = DataLoader(
    train_dataset,
    batch_size=BATCH_SIZE,
    shuffle=True,
    num_workers=0,
    pin_memory=True
)


val_loader = DataLoader(
    val_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=0,
    pin_memory=True
)


test_loader = DataLoader(
    test_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=0,
    pin_memory=True
)


print("\nDataset sizes:")
print("Train:", len(train_dataset))
print("Val  :", len(val_dataset))
print("Test :", len(test_dataset))


print("\nBatch counts:")
print("Train:", len(train_loader))
print("Val  :", len(val_loader))
print("Test :", len(test_loader))

Created dataset: 1355 temporal windows from 101 events
Created dataset: 236 temporal windows from 21 events
Created dataset: 370 temporal windows from 24 events

Dataset sizes:
Train: 1355
Val  : 236
Test : 370

Batch counts:
Train: 678
Val  : 118
Test : 185


In [23]:
print("Checking validation dataset...\n")


for i in range(
    len(val_dataset)
):

    X, Y = val_dataset[i]


    assert X.shape == (
        10,
        8,
        256,
        256
    ), (
        f"Bad X shape at {i}: "
        f"{X.shape}"
    )


    assert Y.shape == (
        10,
        256,
        256
    ), (
        f"Bad Y shape at {i}: "
        f"{Y.shape}"
    )


    assert torch.isfinite(X).all()


    assert torch.isfinite(Y).all()


    unique_values = torch.unique(Y)

    assert all(
        value.item() in [0.0, 1.0]
        for value in unique_values
    )


print(
    "✅ ALL validation samples passed."
)

Checking validation dataset...

✅ ALL validation samples passed.


In [24]:
class DoubleConv(nn.Module):

    def __init__(
        self,
        in_channels,
        out_channels
    ):

        super().__init__()

        self.block = nn.Sequential(

            nn.Conv2d(
                in_channels,
                out_channels,
                kernel_size=3,
                padding=1,
                bias=False
            ),

            nn.BatchNorm2d(
                out_channels
            ),

            nn.ReLU(
                inplace=True
            ),


            nn.Conv2d(
                out_channels,
                out_channels,
                kernel_size=3,
                padding=1,
                bias=False
            ),

            nn.BatchNorm2d(
                out_channels
            ),

            nn.ReLU(
                inplace=True
            )
        )


    def forward(self, x):

        return self.block(x)


class UNet2D(nn.Module):

    def __init__(
        self,
        in_channels=8,
        out_channels=1
    ):

        super().__init__()


        self.enc1 = DoubleConv(
            in_channels,
            32
        )

        self.enc2 = DoubleConv(
            32,
            64
        )

        self.enc3 = DoubleConv(
            64,
            128
        )


        self.bottleneck = DoubleConv(
            128,
            256
        )


        self.pool = nn.MaxPool2d(2)


        self.up3 = nn.ConvTranspose2d(
            256,
            128,
            kernel_size=2,
            stride=2
        )

        self.dec3 = DoubleConv(
            256,
            128
        )


        self.up2 = nn.ConvTranspose2d(
            128,
            64,
            kernel_size=2,
            stride=2
        )

        self.dec2 = DoubleConv(
            128,
            64
        )


        self.up1 = nn.ConvTranspose2d(
            64,
            32,
            kernel_size=2,
            stride=2
        )

        self.dec1 = DoubleConv(
            64,
            32
        )


        self.final = nn.Conv2d(
            32,
            out_channels,
            kernel_size=1
        )


    def forward(self, x):

        e1 = self.enc1(x)

        e2 = self.enc2(
            self.pool(e1)
        )

        e3 = self.enc3(
            self.pool(e2)
        )

        b = self.bottleneck(
            self.pool(e3)
        )


        d3 = self.up3(b)

        d3 = torch.cat(
            [d3, e3],
            dim=1
        )

        d3 = self.dec3(d3)


        d2 = self.up2(d3)

        d2 = torch.cat(
            [d2, e2],
            dim=1
        )

        d2 = self.dec2(d2)


        d1 = self.up1(d2)

        d1 = torch.cat(
            [d1, e1],
            dim=1
        )

        d1 = self.dec1(d1)


        return self.final(d1)


class TemporalUNet(nn.Module):

    def __init__(self):

        super().__init__()

        self.unet = UNet2D(
            in_channels=8,
            out_channels=1
        )


    def forward(self, x):

        B, T, C, H, W = x.shape


        x = x.reshape(
            B * T,
            C,
            H,
            W
        )


        logits = self.unet(x)


        logits = logits.squeeze(1)


        logits = logits.reshape(
            B,
            T,
            H,
            W
        )


        return logits

In [25]:
device = torch.device(
    "cuda"
    if torch.cuda.is_available()
    else "cpu"
)


model = TemporalUNet().to(device)


print("Device:", device)

if torch.cuda.is_available():

    print(
        "GPU:",
        torch.cuda.get_device_name(0)
    )


total_params = sum(
    p.numel()
    for p in model.parameters()
)


print(
    "Total parameters:",
    f"{total_params:,}"
)

Device: cuda
GPU: Tesla T4
Total parameters: 1,928,449


In [26]:
class DiceLoss(nn.Module):

    def __init__(
        self,
        smooth=1.0
    ):

        super().__init__()

        self.smooth = smooth


    def forward(
        self,
        logits,
        targets
    ):

        probs = torch.sigmoid(
            logits
        )


        probs = probs.reshape(-1)

        targets = targets.reshape(-1)


        intersection = (
            probs * targets
        ).sum()


        dice = (
            2.0 * intersection +
            self.smooth
        ) / (
            probs.sum() +
            targets.sum() +
            self.smooth
        )


        return 1.0 - dice


class BCEDiceLoss(nn.Module):

    def __init__(
        self,
        bce_weight=0.5,
        dice_weight=0.5
    ):

        super().__init__()

        self.bce_weight = bce_weight

        self.dice_weight = dice_weight

        self.bce = nn.BCEWithLogitsLoss()

        self.dice = DiceLoss()


    def forward(
        self,
        logits,
        targets
    ):

        bce_loss = self.bce(
            logits,
            targets
        )


        dice_loss = self.dice(
            logits,
            targets
        )


        return (
            self.bce_weight * bce_loss
            +
            self.dice_weight * dice_loss
        )


criterion = BCEDiceLoss()

print(criterion)

BCEDiceLoss(
  (bce): BCEWithLogitsLoss()
  (dice): DiceLoss()
)


In [27]:
def calculate_metrics(
    logits,
    targets,
    threshold=0.5
):

    probs = torch.sigmoid(
        logits
    )


    predictions = (
        probs >= threshold
    )


    targets = targets.bool()


    predictions = predictions.reshape(-1)

    targets = targets.reshape(-1)


    tp = (
        predictions &
        targets
    ).sum().item()


    fp = (
        predictions &
        (~targets)
    ).sum().item()


    fn = (
        (~predictions) &
        targets
    ).sum().item()


    precision = (
        tp /
        (tp + fp + 1e-8)
    )


    recall = (
        tp /
        (tp + fn + 1e-8)
    )


    f1 = (
        2 * precision * recall /
        (precision + recall + 1e-8)
    )


    iou = (
        tp /
        (tp + fp + fn + 1e-8)
    )


    dice = (
        2 * tp /
        (2 * tp + fp + fn + 1e-8)
    )


    return {
        "precision": precision,
        "recall": recall,
        "f1": f1,
        "iou": iou,
        "dice": dice,
        "tp": tp,
        "fp": fp,
        "fn": fn
    }

In [28]:
def evaluate_loader(
    model,
    loader,
    criterion,
    device
):

    model.eval()


    total_loss = 0.0

    total_tp = 0
    total_fp = 0
    total_fn = 0


    with torch.no_grad():

        for X, Y in loader:

            X = X.to(
                device,
                non_blocking=True
            )

            Y = Y.to(
                device,
                non_blocking=True
            )


            logits = model(X)


            loss = criterion(
                logits,
                Y
            )


            total_loss += (
                loss.item() *
                X.size(0)
            )


            predictions = (
                torch.sigmoid(logits)
                >= 0.5
            )


            total_tp += int(
                (
                    predictions &
                    (Y == 1)
                ).sum().item()
            )


            total_fp += int(
                (
                    predictions &
                    (Y == 0)
                ).sum().item()
            )


            total_fn += int(
                (
                    (~predictions) &
                    (Y == 1)
                ).sum().item()
            )


    precision = (
        total_tp /
        (total_tp + total_fp + 1e-8)
    )


    recall = (
        total_tp /
        (total_tp + total_fn + 1e-8)
    )


    f1 = (
        2 * precision * recall /
        (precision + recall + 1e-8)
    )


    iou = (
        total_tp /
        (
            total_tp +
            total_fp +
            total_fn +
            1e-8
        )
    )


    dice = (
        2 * total_tp /
        (
            2 * total_tp +
            total_fp +
            total_fn +
            1e-8
        )
    )


    avg_loss = (
        total_loss /
        len(loader.dataset)
    )


    return {
        "loss": avg_loss,
        "precision": precision,
        "recall": recall,
        "f1": f1,
        "iou": iou,
        "dice": dice,
        "tp": total_tp,
        "fp": total_fp,
        "fn": total_fn
    }

In [31]:
optimizer = optim.AdamW(
    model.parameters(),
    lr=1e-3,
    weight_decay=1e-4
)

print(optimizer)

NUM_EPOCHS = 20
PATIENCE = 5

BEST_MODEL_PATH = Path(
    "/kaggle/working/ts_satfire_unet_best.pt"
)

HISTORY_JSON = Path(
    "/kaggle/working/ts_satfire_training_history.json"
)

HISTORY_CSV = Path(
    "/kaggle/working/ts_satfire_training_history.csv"
)


history = {
    "epoch": [],
    "train_loss": [],
    "val_loss": [],
    "val_precision": [],
    "val_recall": [],
    "val_f1": [],
    "val_iou": [],
    "val_dice": [],
    "val_tp": [],
    "val_fp": [],
    "val_fn": [],
    "epoch_time_seconds": []
}


best_val_f1 = -1.0
best_epoch = 0
epochs_without_improvement = 0


for epoch in range(
    1,
    NUM_EPOCHS + 1
):

    epoch_start = time.time()

    model.train()

    running_loss = 0.0


    # =====================================================
    # TRAIN
    # =====================================================

    for batch_idx, (X, Y) in enumerate(
        train_loader
    ):

        X = X.to(
            device,
            non_blocking=True
        )

        Y = Y.to(
            device,
            non_blocking=True
        )


        optimizer.zero_grad(
            set_to_none=True
        )


        logits = model(X)


        loss = criterion(
            logits,
            Y
        )


        loss.backward()


        optimizer.step()


        running_loss += (
            loss.item() *
            X.size(0)
        )


    train_loss = (
        running_loss /
        len(train_loader.dataset)
    )


    # =====================================================
    # VALIDATION
    # =====================================================

    val_metrics = evaluate_loader(
        model,
        val_loader,
        criterion,
        device
    )


    epoch_time = (
        time.time() -
        epoch_start
    )


    # =====================================================
    # HISTORY
    # =====================================================

    history["epoch"].append(epoch)

    history["train_loss"].append(
        train_loss
    )

    history["val_loss"].append(
        val_metrics["loss"]
    )

    history["val_precision"].append(
        val_metrics["precision"]
    )

    history["val_recall"].append(
        val_metrics["recall"]
    )

    history["val_f1"].append(
        val_metrics["f1"]
    )

    history["val_iou"].append(
        val_metrics["iou"]
    )

    history["val_dice"].append(
        val_metrics["dice"]
    )

    history["val_tp"].append(
        val_metrics["tp"]
    )

    history["val_fp"].append(
        val_metrics["fp"]
    )

    history["val_fn"].append(
        val_metrics["fn"]
    )

    history["epoch_time_seconds"].append(
        epoch_time
    )


    # =====================================================
    # SAVE HISTORY
    # =====================================================

    with open(
        HISTORY_JSON,
        "w"
    ) as f:

        json.dump(
            history,
            f,
            indent=2
        )


    with open(
        HISTORY_CSV,
        "w",
        newline=""
    ) as f:

        writer = csv.writer(f)


        writer.writerow([
            "epoch",
            "train_loss",
            "val_loss",
            "val_precision",
            "val_recall",
            "val_f1",
            "val_iou",
            "val_dice",
            "val_tp",
            "val_fp",
            "val_fn",
            "epoch_time_seconds"
        ])


        for i in range(
            len(history["epoch"])
        ):

            writer.writerow([
                history["epoch"][i],
                history["train_loss"][i],
                history["val_loss"][i],
                history["val_precision"][i],
                history["val_recall"][i],
                history["val_f1"][i],
                history["val_iou"][i],
                history["val_dice"][i],
                history["val_tp"][i],
                history["val_fp"][i],
                history["val_fn"][i],
                history["epoch_time_seconds"][i]
            ])


    # =====================================================
    # PRINT
    # =====================================================

    print("\n" + "=" * 70)

    print(
        f"Epoch {epoch:02d}/{NUM_EPOCHS}"
        f" | {epoch_time / 60:.2f} min"
    )

    print(
        f"Train Loss: "
        f"{train_loss:.5f}"
    )

    print(
        f"Val Loss: "
        f"{val_metrics['loss']:.5f}"
    )

    print(
        f"Precision: "
        f"{val_metrics['precision']:.5f}"
    )

    print(
        f"Recall: "
        f"{val_metrics['recall']:.5f}"
    )

    print(
        f"F1: "
        f"{val_metrics['f1']:.5f}"
    )

    print(
        f"IoU: "
        f"{val_metrics['iou']:.5f}"
    )

    print(
        f"Dice: "
        f"{val_metrics['dice']:.5f}"
    )

    print(
        f"TP: {val_metrics['tp']:,} | "
        f"FP: {val_metrics['fp']:,} | "
        f"FN: {val_metrics['fn']:,}"
    )


    # =====================================================
    # BEST MODEL
    # =====================================================

    if (
        val_metrics["f1"] >
        best_val_f1
    ):

        best_val_f1 = (
            val_metrics["f1"]
        )

        best_epoch = epoch

        epochs_without_improvement = 0


        torch.save(
            {
                "epoch": epoch,
                "model_state_dict":
                    model.state_dict(),
                "optimizer_state_dict":
                    optimizer.state_dict(),
                "val_f1":
                    best_val_f1,
                "history":
                    history
            },
            BEST_MODEL_PATH
        )


        print(
            f"🔥 NEW BEST MODEL"
            f" | Val F1 = "
            f"{best_val_f1:.5f}"
        )


    else:

        epochs_without_improvement += 1


        print(
            f"No improvement "
            f"({epochs_without_improvement}/"
            f"{PATIENCE})"
        )


    # =====================================================
    # EARLY STOPPING
    # =====================================================

    if (
        epochs_without_improvement
        >= PATIENCE
    ):

        print(
            "\n🛑 Early stopping triggered."
        )

        break


print("\n" + "=" * 70)
print("TRAINING COMPLETE")
print("=" * 70)

print(
    "Best epoch:",
    best_epoch
)

print(
    "Best validation F1:",
    f"{best_val_f1:.5f}"
)

print(
    "Best model:",
    BEST_MODEL_PATH
)

AdamW (
Parameter Group 0
    amsgrad: False
    betas: (0.9, 0.999)
    capturable: False
    decoupled_weight_decay: True
    differentiable: False
    eps: 1e-08
    foreach: None
    fused: None
    lr: 0.001
    maximize: False
    weight_decay: 0.0001
)


KeyboardInterrupt: 

In [32]:
# Check whether the previous best checkpoint survived the Kaggle interruption.
import os
import glob

print("Working directory files:")
for f in glob.glob("/kaggle/working/*"):
    print(f, f"{os.path.getsize(f)/1024**2:.2f} MB")

print("\nBest checkpoint exists:",
      os.path.exists("/kaggle/working/ts_satfire_unet_best.pt"))

Working directory files:
/kaggle/working/ts_satfire_unet_best.pt 22.15 MB
/kaggle/working/ts_satfire_training_history.csv 0.00 MB
/kaggle/working/ts_satfire_training_history.json 0.00 MB
/kaggle/working/ts_satfire_split.json 0.00 MB

Best checkpoint exists: True


In [33]:
# Verify the saved best checkpoint.
import os
import torch

CHECKPOINT_PATH = "/kaggle/working/ts_satfire_unet_best.pt"

print("Checkpoint exists:", os.path.exists(CHECKPOINT_PATH))
print("Checkpoint size:",
      f"{os.path.getsize(CHECKPOINT_PATH) / (1024**2):.2f} MB")

checkpoint = torch.load(
    CHECKPOINT_PATH,
    map_location="cpu",
    weights_only=False
)

print("\nCheckpoint type:", type(checkpoint))

if isinstance(checkpoint, dict):
    print("\nCheckpoint keys:")
    for key in checkpoint.keys():
        print(" -", key)

    # Show useful metadata if present.
    for key in ["epoch", "best_val_f1", "val_f1"]:
        if key in checkpoint:
            print(f"{key}: {checkpoint[key]}")
else:
    print("Checkpoint contains a raw model state_dict.")

Checkpoint exists: True
Checkpoint size: 22.15 MB

Checkpoint type: <class 'dict'>

Checkpoint keys:
 - epoch
 - model_state_dict
 - optimizer_state_dict
 - val_f1
 - history
epoch: 4
val_f1: 0.7873871369670997


In [36]:
# ============================================================
# RESTORE EPOCH-4 CHECKPOINT
# ============================================================

import torch
import os

CHECKPOINT_PATH = "/kaggle/working/ts_satfire_unet_best.pt"
RESUME_PATH = "/kaggle/working/ts_satfire_resume.pt"

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# Create the model exactly as originally defined
model = TemporalUNet().to(device)

# Create the same optimizer
optimizer = torch.optim.AdamW(
    model.parameters(),
    lr=1e-3,
    weight_decay=1e-4
)

# Load checkpoint
checkpoint = torch.load(
    CHECKPOINT_PATH,
    map_location=device,
    weights_only=False
)

# Restore model + optimizer
model.load_state_dict(checkpoint["model_state_dict"])
optimizer.load_state_dict(checkpoint["optimizer_state_dict"])

# Restore training information
start_epoch = checkpoint["epoch"] + 1
best_val_f1 = checkpoint["val_f1"]
history = checkpoint["history"]

print("=" * 60)
print("✅ CHECKPOINT RESTORED")
print("=" * 60)

print(f"Saved epoch:       {checkpoint['epoch']}")
print(f"Next epoch:        {start_epoch}")
print(f"Best Val F1:       {best_val_f1:.5f}")
print(f"History entries:   {len(history)}")
print(f"Device:            {device}")

print("\nModel weights:      ✅")
print("Optimizer state:   ✅")
print("Training history:  ✅")

assert start_epoch == 5, "Expected to resume from Epoch 5"
assert abs(best_val_f1 - 0.7873871369670997) < 1e-6

print("\n🔥 READY TO RESUME FROM EPOCH 5")

✅ CHECKPOINT RESTORED
Saved epoch:       4
Next epoch:        5
Best Val F1:       0.78739
History entries:   12
Device:            cuda

Model weights:      ✅
Optimizer state:   ✅
Training history:  ✅

🔥 READY TO RESUME FROM EPOCH 5


In [38]:
print("History type:", type(history))

if isinstance(history, dict):
    print("\nHistory keys:")
    for key in history.keys():
        print(" -", key)

    print("\nHistory contents:")
    for key, value in history.items():
        if isinstance(value, list):
            print(f"{key}: list with {len(value)} entries")
        else:
            print(f"{key}: {type(value)}")
else:
    print("History length:", len(history))
    print("First entry:", history[0] if len(history) > 0 else "EMPTY")

History type: <class 'dict'>

History keys:
 - epoch
 - train_loss
 - val_loss
 - val_precision
 - val_recall
 - val_f1
 - val_iou
 - val_dice
 - val_tp
 - val_fp
 - val_fn
 - epoch_time_seconds

History contents:
epoch: list with 4 entries
train_loss: list with 4 entries
val_loss: list with 4 entries
val_precision: list with 4 entries
val_recall: list with 4 entries
val_f1: list with 4 entries
val_iou: list with 4 entries
val_dice: list with 4 entries
val_tp: list with 4 entries
val_fp: list with 4 entries
val_fn: list with 4 entries
epoch_time_seconds: list with 4 entries


In [39]:
# ============================================================
# RELOAD SAFE EPOCH-4 CHECKPOINT
# ============================================================

import torch

CHECKPOINT_PATH = "/kaggle/working/ts_satfire_unet_best.pt"

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# Create the same architecture
model = TemporalUNet().to(device)

# Create the same optimizer
optimizer = torch.optim.AdamW(
    model.parameters(),
    lr=1e-3,
    weight_decay=1e-4
)

# Load the untouched Epoch-4 checkpoint
checkpoint = torch.load(
    CHECKPOINT_PATH,
    map_location=device,
    weights_only=False
)

model.load_state_dict(checkpoint["model_state_dict"])
optimizer.load_state_dict(checkpoint["optimizer_state_dict"])

start_epoch = checkpoint["epoch"] + 1
best_val_f1 = checkpoint["val_f1"]
history = checkpoint["history"]

print("=" * 60)
print("✅ SAFE CHECKPOINT RELOADED")
print("=" * 60)

print(f"Saved epoch:      {checkpoint['epoch']}")
print(f"Starting epoch:   {start_epoch}")
print(f"Best Val F1:      {best_val_f1:.5f}")
print(f"History type:     {type(history)}")
print(f"History epochs:   {history['epoch']}")
print(f"Device:           {device}")

assert checkpoint["epoch"] == 4
assert start_epoch == 5
assert abs(best_val_f1 - 0.7873871369670997) < 1e-6

print("\n🔥 READY FOR EPOCH 5")

✅ SAFE CHECKPOINT RELOADED
Saved epoch:      4
Starting epoch:   5
Best Val F1:      0.78739
History type:     <class 'dict'>
History epochs:   [1, 2, 3, 4]
Device:           cuda

🔥 READY FOR EPOCH 5


In [40]:
# ============================================================
# CRASH-SAFE TRAINING LOOP
# RESUME FROM EPOCH 5
# ============================================================

import time
import json
import csv
import torch

MAX_EPOCHS = 20
PATIENCE = 5

RESUME_PATH = "/kaggle/working/ts_satfire_resume.pt"
BEST_PATH = "/kaggle/working/ts_satfire_unet_best.pt"

# ------------------------------------------------------------
# Determine patience state from existing history
# ------------------------------------------------------------
if len(history["val_f1"]) > 0:

    last_f1 = history["val_f1"][-1]

    # Number of consecutive epochs without improvement
    patience_counter = 0

    for f1_value in reversed(history["val_f1"]):

        if f1_value > best_val_f1:
            break

        patience_counter += 1

else:
    patience_counter = 0

print("=" * 70)
print("🔥 RESUMING TRAINING")
print("=" * 70)
print(f"Starting epoch: {start_epoch}")
print(f"Best Val F1:    {best_val_f1:.5f}")
print(f"Max epochs:     {MAX_EPOCHS}")
print(f"Patience:       {PATIENCE}")
print("=" * 70)


for epoch in range(start_epoch, MAX_EPOCHS + 1):

    epoch_start = time.time()

    # ========================================================
    # TRAIN
    # ========================================================

    model.train()

    train_loss_sum = 0.0
    train_batches = 0

    for X, Y in train_loader:

        X = X.to(device, non_blocking=True)
        Y = Y.to(device, non_blocking=True)

        optimizer.zero_grad(set_to_none=True)

        logits = model(X)

        loss = criterion(logits, Y)

        loss.backward()

        optimizer.step()

        train_loss_sum += loss.item()
        train_batches += 1

    train_loss = train_loss_sum / train_batches


    # ========================================================
    # VALIDATION
    # ========================================================

    model.eval()

    val_loss_sum = 0.0
    val_batches = 0

    TP = 0
    FP = 0
    FN = 0

    with torch.no_grad():

        for X, Y in val_loader:

            X = X.to(device, non_blocking=True)
            Y = Y.to(device, non_blocking=True)

            logits = model(X)

            loss = criterion(logits, Y)

            val_loss_sum += loss.item()
            val_batches += 1

            probs = torch.sigmoid(logits)

            preds = probs >= 0.5
            targets = Y >= 0.5

            TP += torch.logical_and(
                preds, targets
            ).sum().item()

            FP += torch.logical_and(
                preds, ~targets
            ).sum().item()

            FN += torch.logical_and(
                ~preds, targets
            ).sum().item()

    val_loss = val_loss_sum / val_batches


    # ========================================================
    # METRICS
    # ========================================================

    precision = TP / (TP + FP + 1e-8)

    recall = TP / (TP + FN + 1e-8)

    f1 = (
        2 * precision * recall /
        (precision + recall + 1e-8)
    )

    iou = TP / (
        TP + FP + FN + 1e-8
    )

    dice = (
        2 * TP /
        (2 * TP + FP + FN + 1e-8)
    )

    epoch_time = time.time() - epoch_start


    # ========================================================
    # APPEND TO EXISTING HISTORY DICTIONARY
    # ========================================================

    history["epoch"].append(epoch)
    history["train_loss"].append(train_loss)
    history["val_loss"].append(val_loss)
    history["val_precision"].append(precision)
    history["val_recall"].append(recall)
    history["val_f1"].append(f1)
    history["val_iou"].append(iou)
    history["val_dice"].append(dice)
    history["val_tp"].append(TP)
    history["val_fp"].append(FP)
    history["val_fn"].append(FN)
    history["epoch_time_seconds"].append(epoch_time)


    # ========================================================
    # PRINT RESULTS
    # ========================================================

    print("\n" + "=" * 70)

    print(
        f"Epoch {epoch:02d}/{MAX_EPOCHS} | "
        f"{epoch_time / 60:.2f} min"
    )

    print(f"Train Loss: {train_loss:.5f}")
    print(f"Val Loss: {val_loss:.5f}")
    print(f"Precision: {precision:.5f}")
    print(f"Recall: {recall:.5f}")
    print(f"F1: {f1:.5f}")
    print(f"IoU: {iou:.5f}")
    print(f"Dice: {dice:.5f}")

    print(
        f"TP: {TP:,} | "
        f"FP: {FP:,} | "
        f"FN: {FN:,}"
    )


    # ========================================================
    # BEST MODEL
    # ========================================================

    if f1 > best_val_f1:

        best_val_f1 = f1
        patience_counter = 0

        torch.save(
            {
                "epoch": epoch,
                "model_state_dict": model.state_dict(),
                "optimizer_state_dict": optimizer.state_dict(),
                "val_f1": best_val_f1,
                "history": history
            },
            BEST_PATH
        )

        print(
            f"🔥 NEW BEST MODEL | "
            f"Val F1 = {best_val_f1:.5f}"
        )

    else:

        patience_counter += 1

        print(
            f"No improvement "
            f"({patience_counter}/{PATIENCE})"
        )


    # ========================================================
    # SAVE RESUME CHECKPOINT
    # ========================================================

    torch.save(
        {
            "epoch": epoch,
            "model_state_dict": model.state_dict(),
            "optimizer_state_dict": optimizer.state_dict(),
            "val_f1": best_val_f1,
            "history": history,
            "patience_counter": patience_counter
        },
        RESUME_PATH
    )

    print(
        f"💾 Resume checkpoint saved "
        f"after Epoch {epoch}"
    )


    # ========================================================
    # SAVE JSON HISTORY
    # ========================================================

    with open(
        "/kaggle/working/ts_satfire_training_history.json",
        "w"
    ) as f:

        json.dump(history, f, indent=2)


    # ========================================================
    # SAVE CSV HISTORY
    # ========================================================

    fieldnames = [
        "epoch",
        "train_loss",
        "val_loss",
        "val_precision",
        "val_recall",
        "val_f1",
        "val_iou",
        "val_dice",
        "val_tp",
        "val_fp",
        "val_fn",
        "epoch_time_seconds"
    ]

    with open(
        "/kaggle/working/ts_satfire_training_history.csv",
        "w",
        newline=""
    ) as f:

        writer = csv.DictWriter(
            f,
            fieldnames=fieldnames
        )

        writer.writeheader()

        for i in range(len(history["epoch"])):

            row = {
                key: history[key][i]
                for key in fieldnames
            }

            writer.writerow(row)


    # ========================================================
    # EARLY STOPPING
    # ========================================================

    if patience_counter >= PATIENCE:

        print("\n🛑 EARLY STOPPING")

        print(
            f"Best Val F1 = "
            f"{best_val_f1:.5f}"
        )

        break


print("\n" + "=" * 70)
print("TRAINING LOOP FINISHED")
print("=" * 70)
print(f"Best Val F1: {best_val_f1:.5f}")

🔥 RESUMING TRAINING
Starting epoch: 5
Best Val F1:    0.78739
Max epochs:     20
Patience:       5

Epoch 05/20 | 34.88 min
Train Loss: 0.12849
Val Loss: 0.17017
Precision: 0.89945
Recall: 0.66377
F1: 0.76385
IoU: 0.61792
Dice: 0.76385
TP: 647,119 | FP: 72,338 | FN: 327,791
No improvement (5/5)
💾 Resume checkpoint saved after Epoch 5

🛑 EARLY STOPPING
Best Val F1 = 0.78739

TRAINING LOOP FINISHED
Best Val F1: 0.78739


In [41]:
# ============================================================
# RESTORE LATEST TRAINING STATE
# CURRENT STATE: EPOCH 5 COMPLETED
# NEXT: EPOCH 6
# ============================================================

import torch
import os

BEST_PATH = "/kaggle/working/ts_satfire_unet_best.pt"
RESUME_PATH = "/kaggle/working/ts_satfire_resume.pt"

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# ------------------------------------------------------------
# Create the exact same model architecture
# ------------------------------------------------------------

model = TemporalUNet().to(device)

# ------------------------------------------------------------
# Create the same optimizer
# ------------------------------------------------------------

optimizer = torch.optim.AdamW(
    model.parameters(),
    lr=1e-3,
    weight_decay=1e-4
)

# ------------------------------------------------------------
# Load the LATEST completed training state
# This is Epoch 5, NOT the best-model checkpoint.
# ------------------------------------------------------------

resume_checkpoint = torch.load(
    RESUME_PATH,
    map_location=device,
    weights_only=False
)

model.load_state_dict(
    resume_checkpoint["model_state_dict"]
)

optimizer.load_state_dict(
    resume_checkpoint["optimizer_state_dict"]
)

# ------------------------------------------------------------
# Restore training state
# ------------------------------------------------------------

last_completed_epoch = resume_checkpoint["epoch"]

start_epoch = last_completed_epoch + 1

history = resume_checkpoint["history"]

# IMPORTANT:
# The best F1 stored in the resume checkpoint is the BEST
# F1 achieved so far, not Epoch-5's F1.
best_val_f1 = resume_checkpoint["val_f1"]

# Epoch 5 did not improve Epoch 4.
# Therefore the correct patience count is 1.
patience_counter = 1

print("=" * 70)
print("✅ LATEST TRAINING CHECKPOINT RESTORED")
print("=" * 70)

print(f"Last completed epoch: {last_completed_epoch}")
print(f"Next epoch:           {start_epoch}")
print(f"Best Val F1:          {best_val_f1:.5f}")
print(f"Patience:             {patience_counter}/5")
print(f"History epochs:       {history['epoch']}")
print(f"Device:               {device}")

# ------------------------------------------------------------
# Safety checks
# ------------------------------------------------------------

assert last_completed_epoch == 5
assert start_epoch == 6
assert abs(best_val_f1 - 0.7873871369670997) < 1e-6

print("\n🔥 CORRECT STATE CONFIRMED")
print("Epoch 5 will NOT be retrained.")
print("Next training epoch = Epoch 6")

✅ LATEST TRAINING CHECKPOINT RESTORED
Last completed epoch: 5
Next epoch:           6
Best Val F1:          0.78739
Patience:             1/5
History epochs:       [1, 2, 3, 4, 5]
Device:               cuda

🔥 CORRECT STATE CONFIRMED
Epoch 5 will NOT be retrained.
Next training epoch = Epoch 6


In [ ]:
# ============================================================
# CRASH-SAFE TRAINING LOOP
# STARTING FROM EPOCH 6
# ============================================================

import time
import json
import csv
import torch

MAX_EPOCHS = 20
PATIENCE = 5

BEST_PATH = "/kaggle/working/ts_satfire_unet_best.pt"
RESUME_PATH = "/kaggle/working/ts_satfire_resume.pt"

print("=" * 70)
print("🔥 CONTINUING TRAINING")
print("=" * 70)
print(f"Starting epoch: {start_epoch}")
print(f"Best Val F1:    {best_val_f1:.5f}")
print(f"Patience:       {patience_counter}/{PATIENCE}")
print("=" * 70)


for epoch in range(start_epoch, MAX_EPOCHS + 1):

    epoch_start = time.time()

    # ========================================================
    # TRAIN
    # ========================================================

    model.train()

    train_loss_sum = 0.0
    train_batches = 0

    for X, Y in train_loader:

        X = X.to(device, non_blocking=True)
        Y = Y.to(device, non_blocking=True)

        optimizer.zero_grad(set_to_none=True)

        logits = model(X)

        loss = criterion(logits, Y)

        loss.backward()

        optimizer.step()

        train_loss_sum += loss.item()
        train_batches += 1

    train_loss = train_loss_sum / train_batches

    # ========================================================
    # VALIDATION
    # ========================================================

    model.eval()

    val_loss_sum = 0.0
    val_batches = 0

    TP = 0
    FP = 0
    FN = 0

    with torch.no_grad():

        for X, Y in val_loader:

            X = X.to(device, non_blocking=True)
            Y = Y.to(device, non_blocking=True)

            logits = model(X)

            loss = criterion(logits, Y)

            val_loss_sum += loss.item()
            val_batches += 1

            probs = torch.sigmoid(logits)

            preds = probs >= 0.5
            targets = Y >= 0.5

            TP += torch.logical_and(
                preds,
                targets
            ).sum().item()

            FP += torch.logical_and(
                preds,
                ~targets
            ).sum().item()

            FN += torch.logical_and(
                ~preds,
                targets
            ).sum().item()

    val_loss = val_loss_sum / val_batches

    # ========================================================
    # METRICS
    # ========================================================

    precision = TP / (TP + FP + 1e-8)

    recall = TP / (TP + FN + 1e-8)

    f1 = (
        2 * precision * recall /
        (precision + recall + 1e-8)
    )

    iou = TP / (
        TP + FP + FN + 1e-8
    )

    dice = (
        2 * TP /
        (2 * TP + FP + FN + 1e-8)
    )

    epoch_time = time.time() - epoch_start

    # ========================================================
    # ADD NEW RESULTS TO EXISTING HISTORY DICTIONARY
    # ========================================================

    history["epoch"].append(epoch)
    history["train_loss"].append(train_loss)
    history["val_loss"].append(val_loss)
    history["val_precision"].append(precision)
    history["val_recall"].append(recall)
    history["val_f1"].append(f1)
    history["val_iou"].append(iou)
    history["val_dice"].append(dice)
    history["val_tp"].append(TP)
    history["val_fp"].append(FP)
    history["val_fn"].append(FN)
    history["epoch_time_seconds"].append(epoch_time)

    # ========================================================
    # PRINT RESULTS
    # ========================================================

    print("\n" + "=" * 70)

    print(
        f"Epoch {epoch:02d}/{MAX_EPOCHS} | "
        f"{epoch_time / 60:.2f} min"
    )

    print(f"Train Loss: {train_loss:.5f}")
    print(f"Val Loss:   {val_loss:.5f}")
    print(f"Precision:  {precision:.5f}")
    print(f"Recall:     {recall:.5f}")
    print(f"F1:         {f1:.5f}")
    print(f"IoU:        {iou:.5f}")
    print(f"Dice:       {dice:.5f}")

    print(
        f"TP: {TP:,} | "
        f"FP: {FP:,} | "
        f"FN: {FN:,}"
    )

    # ========================================================
    # CHECK FOR NEW BEST
    # ========================================================

    if f1 > best_val_f1:

        best_val_f1 = f1

        patience_counter = 0

        torch.save(
            {
                "epoch": epoch,
                "model_state_dict": model.state_dict(),
                "optimizer_state_dict": optimizer.state_dict(),
                "val_f1": best_val_f1,
                "history": history
            },
            BEST_PATH
        )

        print(
            f"🔥 NEW BEST MODEL | "
            f"Val F1 = {best_val_f1:.5f}"
        )

    else:

        patience_counter += 1

        print(
            f"No improvement "
            f"({patience_counter}/{PATIENCE})"
        )

    # ========================================================
    # SAVE RESUME CHECKPOINT
    # ========================================================

    torch.save(
        {
            "epoch": epoch,
            "model_state_dict": model.state_dict(),
            "optimizer_state_dict": optimizer.state_dict(),
            "val_f1": best_val_f1,
            "history": history,
            "patience_counter": patience_counter
        },
        RESUME_PATH
    )

    print(
        f"💾 Resume checkpoint saved "
        f"after Epoch {epoch}"
    )

    # ========================================================
    # SAVE JSON HISTORY
    # ========================================================

    with open(
        "/kaggle/working/ts_satfire_training_history.json",
        "w"
    ) as f:

        json.dump(history, f, indent=2)

    # ========================================================
    # SAVE CSV HISTORY
    # ========================================================

    fieldnames = [
        "epoch",
        "train_loss",
        "val_loss",
        "val_precision",
        "val_recall",
        "val_f1",
        "val_iou",
        "val_dice",
        "val_tp",
        "val_fp",
        "val_fn",
        "epoch_time_seconds"
    ]

    with open(
        "/kaggle/working/ts_satfire_training_history.csv",
        "w",
        newline=""
    ) as f:

        writer = csv.DictWriter(
            f,
            fieldnames=fieldnames
        )

        writer.writeheader()

        for i in range(len(history["epoch"])):

            writer.writerow({
                key: history[key][i]
                for key in fieldnames
            })

    # ========================================================
    # EARLY STOPPING
    # ========================================================

    if patience_counter >= PATIENCE:

        print("\n🛑 EARLY STOPPING")

        print(
            f"Best Val F1 = "
            f"{best_val_f1:.5f}"
        )

        break


print("\n" + "=" * 70)
print("TRAINING LOOP FINISHED")
print("=" * 70)
print(f"Best Val F1: {best_val_f1:.5f}")

🔥 CONTINUING TRAINING
Starting epoch: 6
Best Val F1:    0.78739
Patience:       1/5

Epoch 06/20 | 34.78 min
Train Loss: 0.12377
Val Loss:   0.21492
Precision:  0.93326
Recall:     0.55714
F1:         0.69774
IoU:        0.53579
Dice:       0.69774
TP: 543,157 | FP: 38,840 | FN: 431,753
No improvement (2/5)
💾 Resume checkpoint saved after Epoch 6

Epoch 07/20 | 37.46 min
Train Loss: 0.12483
Val Loss:   0.18499
Precision:  0.91844
Recall:     0.62033
F1:         0.74051
IoU:        0.58795
Dice:       0.74051
TP: 604,769 | FP: 53,702 | FN: 370,141
No improvement (3/5)
💾 Resume checkpoint saved after Epoch 7

Epoch 08/20 | 38.74 min
Train Loss: 0.12383
Val Loss:   0.17371
Precision:  0.90679
Recall:     0.65119
F1:         0.75802
IoU:        0.61034
Dice:       0.75802
TP: 634,853 | FP: 65,260 | FN: 340,057
No improvement (4/5)
💾 Resume checkpoint saved after Epoch 8


**NEXT DATASET **

In [ ]:
from pathlib import Path
import shutil

root = Path("/kaggle/input")

print("Disk usage:")
print(shutil.disk_usage("/kaggle"))

print("\nMounted datasets:")
for p in sorted(root.iterdir()):
    print(f"\n{p.name}")

    if p.is_dir():
        count = 0
        sample = []

        for f in p.rglob("*"):
            if f.is_file():
                count += 1
                if len(sample) < 5:
                    sample.append(str(f.relative_to(p)))

        print("File count:", count)
        print("Sample files:", sample)

In [1]:
import rasterio
import numpy as np

# Use a Day raster from an existing dataset sample
sample = train_dataset.samples[0]
day_path = sample["day_files"][0]

with rasterio.open(day_path) as src:
    print("File:", day_path)
    print("Raster shape:", src.shape)
    print("Number of bands:", src.count)
    print("Band descriptions:", src.descriptions)
    print("Band dtypes:", src.dtypes)
    print("NoData values:", src.nodatavals)

    active_fire = src.read(7).astype(np.float32)
    band_mask = src.read_masks(7) > 0

valid = np.isfinite(active_fire) & band_mask
values = active_fire[valid]

print("\nBand 7 diagnostic")
print("Valid pixels:", int(valid.sum()))
print("Invalid/masked pixels:", int((~valid).sum()))

if values.size:
    print("Min:", float(values.min()))
    print("Max:", float(values.max()))
    print("Mean:", float(values.mean()))
    print("Unique values (up to 20):", np.unique(values)[:20])

    positive = values > 0
    print("Positive pixels:", int(positive.sum()))
    print("Positive fraction among valid pixels:",
          float(positive.mean()))
else:
    print("No valid pixels found.")

NameError: name 'train_dataset' is not defined